In [ ]:
import streamlit as st
import pandas as pd
import altair as alt

# Load the data
@st.cache_data
def load_data():
    url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/building_inventory.csv"
    return pd.read_csv(url)

df = load_data()

# Clean and prepare data
df = df.dropna(subset=['Year Constructed', 'Square Footage'])
df['Year Constructed'] = df['Year Constructed'].astype(int)
df['Square Footage'] = df['Square Footage'].astype(int)

# App title
st.title("Illinois State Building Inventory Analysis")
st.write("This app provides two visualizations of the Illinois state building inventory dataset.")

# First Visualization: Building Construction Over Time
st.header("1. Building Construction Timeline")
st.write("""
This line chart shows the number of buildings constructed each year, with an interactive filter 
to select specific agencies. The y-axis uses a linear scale to accurately represent counts, 
while the x-axis shows years. I chose a line chart to emphasize trends over time. The color 
encoding helps distinguish between different agencies. If I had more time, I would add tooltips 
with more detailed information about each data point.
""")

# Agency selection for first chart
selected_agencies = st.multiselect(
    'Select agencies to display:',
    options=df['Agency Name'].unique(),
    default=df['Agency Name'].unique()[:3]
)

# Filter data based on selection
filtered_df = df[df['Agency Name'].isin(selected_agencies)]

# Create construction timeline chart
timeline = alt.Chart(filtered_df).mark_line(point=True).encode(
    x=alt.X('Year Constructed:O', title='Year Constructed'),
    y=alt.Y('count():Q', title='Number of Buildings'),
    color=alt.Color('Agency Name:N', legend=alt.Legend(title="Agency")),
    tooltip=['Agency Name', 'count()']
).properties(
    width=700,
    height=400
).interactive()

st.altair_chart(timeline)

# Second Visualization: Square Footage by Usage
st.header("2. Building Usage Analysis")
st.write("""
This interactive bar chart compares the average square footage by building usage type. 
I used a log scale for the x-axis to better handle the wide range of building sizes. 
The color represents different usage types, making it easy to distinguish categories. 
Clicking on the legend filters the data shown. With more time, I would add a second 
level of filtering by agency or location.
""")

# Prepare data for second chart
usage_df = df.groupby('Usage Description')['Square Footage'].mean().reset_index()

# Create interactive bar chart
bars = alt.Chart(df).mark_bar().encode(
    x=alt.X('mean(Square Footage):Q', scale=alt.Scale(type='log'), title='Average Square Footage (log scale)'),
    y=alt.Y('Usage Description:N', sort='-x', title='Usage Type'),
    color=alt.Color('Usage Description:N', legend=None),
    tooltip=['Usage Description', 'mean(Square Footage)']
).properties(
    width=700,
    height=500
).interactive()

# Add selection
selection = alt.selection_multi(fields=['Usage Description'])
color = alt.condition(selection,
                      alt.Color('Usage Description:N', legend=None),
                      alt.value('lightgray'))

clickable_legend = alt.Chart(df).mark_point().encode(
    y=alt.Y('Usage Description:N', axis=alt.Axis(orient='right'),
    color=color
).add_selection(
    selection
)

chart = alt.hconcat(bars, clickable_legend)
st.altair_chart(chart)

# Write-up about differences from HW5
st.header("Comparison with Homework #5")
st.write("""
"This visualization is significantly different from my Homework #5 submission. 
For HW5, I focused on geographic distribution of buildings, while this assignment 
uses temporal and categorical analyses. The chart types (line and bar) are completely 
new, and the interactivity features (multi-select filtering and clickable legend) 
were not present in my previous work."
""")